In [0]:
dbutils.widgets.text("env", "dev")

In [0]:
%sql
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_incidents AS
SELECT
    date_trunc('HOUR', called_at) AS hour,
    capability, model_config, transport,
    count_if(is_blank_output) AS blank_output_count,
    count(*)                  AS total_successful_calls,
    count_if(is_blank_output) * 1.0 / count(*) AS blank_output_rate
FROM mq_gmdf_dev.oil_obs.v_llm_bronze
WHERE success = true
  AND is_credential_fastfail = false
GROUP BY 1, 2, 3, 4
HAVING count_if(is_blank_output) > 0;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- blank_output_findings — aggregated over 6h window for MERGE dedup.
-- Key on (capability, model_config), NOT the hour — a sustained blank-output condition
-- is one incident. The alert check thresholds (>2% rate, >3 blank, >=10 total) are
-- applied here so the findings table contains only actionable rows.
-- Currently verified as genuinely 0 rows. Wired so it works when it fires.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_findings AS
SELECT
    capability, model_config,
    sum(blank_output_count) AS blank_count_window,
    sum(total_successful_calls) AS total_calls_window,
    round(sum(blank_output_count) * 1.0 / nullif(sum(total_successful_calls), 0), 4) AS blank_rate_window,
    max(hour) AS latest_hour,
    sha2(concat_ws('|',
        coalesce(capability, '<null>'),
        coalesce(model_config, '<null>')
    ), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.blank_output_incidents
WHERE hour >= date_trunc('HOUR', current_timestamp() - INTERVAL 6 HOURS)
GROUP BY capability, model_config
HAVING sum(blank_output_count) > 3
   AND sum(total_successful_calls) >= 10
   AND sum(blank_output_count) * 1.0 / nullif(sum(total_successful_calls), 0) > 0.02;

In [0]:
%sql
-- transport_violations
-- violation_signature (2026-08-27): the finding is the CONFIGURATION, not the call.
-- violation_tier (2026-08-27): an unlisted CAPABILITY running on a transport and model_config
-- already sanctioned for some other capability in this env is a paperwork lag -> 'digest'.
-- A transport or model_config sanctioned for NOTHING in this env is a different claim about
-- the system -> 'immediate'. scheduler_run alone never escalates: 'live' spans two agent
-- generations (§4.2) so it does not discriminate.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.transport_violations AS
WITH env_sanctioned AS (
  -- Every value sanctioned for ANY capability in this environment.
  -- Aggregate with no GROUP BY, so this always yields exactly one row, even when the
  -- allowlist is empty for :env.
  SELECT
      array_distinct(flatten(collect_list(allowed_transports)))     AS ok_transports,
      array_distinct(flatten(collect_list(allowed_model_configs)))  AS ok_model_configs,
      array_distinct(flatten(collect_list(allowed_scheduler_runs))) AS ok_scheduler_runs
  FROM mq_gmdf_dev.oil_obs.runtime_allowlist
  WHERE environment = :env
),
flagged AS (
  SELECT
      b.id, b.shift_date, b.shift_type, b.batch_nbr,
      b.capability, b.scheduler_run, b.transport, b.model_config, b.called_at,
      CASE
        WHEN a.capability IS NULL                                          THEN 'unknown_capability'
        WHEN NOT array_contains(a.allowed_transports,      b.transport)     THEN 'transport_not_allowed'
        WHEN NOT array_contains(a.allowed_model_configs,   b.model_config)  THEN 'model_config_not_allowed'
        WHEN NOT array_contains(a.allowed_scheduler_runs,  b.scheduler_run) THEN 'scheduler_run_not_allowed'
      END AS violation_type
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  LEFT JOIN (SELECT * FROM mq_gmdf_dev.oil_obs.runtime_allowlist WHERE environment = :env) a
         ON a.capability = b.capability
  WHERE b.called_at >= current_timestamp() - INTERVAL 60 MINUTES
    AND (a.capability IS NULL
         OR NOT array_contains(a.allowed_transports,     b.transport)
         OR NOT array_contains(a.allowed_model_configs,  b.model_config)
         OR NOT array_contains(a.allowed_scheduler_runs, b.scheduler_run))
)
SELECT
    f.id, f.shift_date, f.shift_type, f.batch_nbr,
    f.capability, f.scheduler_run, f.transport, f.model_config, f.called_at,
    f.violation_type,
    -- coalesce because concat_ws DROPS nulls: ('a', NULL, 'b') and ('a', 'b', NULL) would
    -- otherwise hash identically. sha2 of a NULL input returns NULL, which would also
    -- silently collapse every such row onto one incident.
    sha2(concat_ws('|',
        coalesce(f.capability,     '<null>'),
        coalesce(f.transport,      '<null>'),
        coalesce(f.model_config,   '<null>'),
        coalesce(f.scheduler_run,  '<null>'),
        coalesce(f.violation_type, '<null>')
    ), 256) AS violation_signature,
    CASE
      -- Empty allowlist for :env makes every value read as unsanctioned. That is a broken
      -- reference table, not a fleet of bad configs — runtime_allowlist_populated owns it.
      -- Fail toward 'digest' so a seed failure cannot page 500 times.
      WHEN size(coalesce(s.ok_transports, array())) = 0                            THEN 'digest'
      WHEN NOT array_contains(s.ok_transports,    coalesce(f.transport,    '<null>')) THEN 'immediate'
      WHEN NOT array_contains(s.ok_model_configs, coalesce(f.model_config, '<null>')) THEN 'immediate'
      ELSE 'digest'
    END AS violation_tier,
    current_timestamp() AS detected_at
FROM flagged f
CROSS JOIN env_sanctioned s;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- transport_violation_signatures — one row per DISTINCT configuration in the window.
-- Exists because obs_incidents MERGEs on (detector, source_row_id): a source with repeated
-- keys throws MULTIPLE_SOURCE_ROWS_MATCHED. The GROUP BY is exactly the sha2 input, so this
-- is guaranteed 1:1 with violation_signature.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.transport_violation_signatures AS
SELECT
    violation_signature,
    violation_tier,
    violation_type,
    capability, transport, model_config, scheduler_run,
    count(*)            AS calls_in_window,
    min(called_at)      AS first_called_at,
    max(called_at)      AS last_called_at,
    max(id)             AS sample_row_id,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.transport_violations
GROUP BY violation_signature, violation_tier, violation_type,
         capability, transport, model_config, scheduler_run;

In [ ]:
%sql
-- response_schema_drift — top-level key comparison via explode.
-- Two earlier approaches failed on this data:
--   1. DDL string equality: nullability variation alone changes the string.
--   2. Regex 'name:' extraction from DDL: matches at every nesting depth, so
--      proposal.op_fields.insert_after_batch was hoisted and read as top-level drift.
-- Exploding a map<string,string> cast yields the top-level key set at the correct depth.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_schema_drift AS
WITH baseline_keys AS (
  SELECT b.capability, k.key AS field_name, count(*) AS baseline_present
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  GROUP BY b.capability, k.key
),
baseline_rows AS (
  SELECT b.capability, count(*) AS n_rows
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  GROUP BY b.capability
),
current_keys AS (
  SELECT b.capability, k.key AS field_name, count(*) AS current_present
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true
  LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability, k.key
),
current_rows AS (
  SELECT b.capability, count(*) AS n_rows
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  WHERE b.success = true AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.called_at >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY b.capability
)
SELECT
    coalesce(bk.capability, ck.capability)          AS capability,
    coalesce(bk.field_name, ck.field_name)          AS field_name,
    bk.baseline_present,
    br.n_rows                                       AS baseline_rows,
    round(bk.baseline_present * 1.0 / br.n_rows, 3) AS baseline_presence_rate,
    coalesce(ck.current_present, 0)                 AS current_present,
    cr.n_rows                                       AS current_rows,
    CASE WHEN bk.field_name IS NULL THEN 'field_added'
         ELSE 'field_missing' END                   AS drift_type,
    true                                            AS schema_changed,
    sha2(concat_ws('|',
        coalesce(coalesce(bk.capability, ck.capability), '<null>'),
        coalesce(coalesce(bk.field_name, ck.field_name), '<null>')
    ), 256) AS finding_signature,
    current_timestamp()                             AS detected_at
FROM      baseline_keys bk
FULL JOIN current_keys  ck ON ck.capability = bk.capability AND ck.field_name = bk.field_name
LEFT JOIN baseline_rows br ON br.capability = coalesce(bk.capability, ck.capability)
LEFT JOIN current_rows  cr ON cr.capability = coalesce(bk.capability, ck.capability)
WHERE cr.n_rows >= 10
  AND (bk.field_name IS NULL
       OR (coalesce(ck.current_present, 0) = 0
           AND bk.baseline_present * 1.0 / br.n_rows >= 0.2));